In [8]:
import pandas as pd

df = pd.read_csv(r"C:\Users\Dell\OneDrive\Desktop\Multimodal Personality-Driven Career Intelligence System\data\personality_dataset\mbti_1.csv")

print(df.shape)

(8675, 2)


In [9]:
df.head()


,type,posts
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...
1,ENTP,'I'm finding the lack of me in these posts ver...
2,INTP,'Good one _____ https://www.youtube.com/wat...
3,INTJ,"'Dear INTP, I enjoyed our conversation the o..."
4,ENTJ,'You're fired.|||That's another silly misconce...


In [10]:
import re

def clean_text(text):
    
    # Split posts separated by |||
    text = text.replace("|||", " ")

    # Remove links
    text = re.sub(r"http\S+", "", text)

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove special characters
    text = re.sub(r"[^a-zA-Z\s]", "", text)

    # Lowercase
    text = text.lower()

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [11]:
df["clean_posts"] = df["posts"].apply(clean_text)


In [12]:
df[["posts", "clean_posts"]].head()


,posts,clean_posts
0,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...,enfp and intj moments sportscenter not top ten...
1,'I'm finding the lack of me in these posts ver...,im finding the lack of me in these posts very ...
2,'Good one _____ https://www.youtube.com/wat...,good one of course to which i say i know thats...
3,"'Dear INTP, I enjoyed our conversation the o...",dear intp i enjoyed our conversation the other...
4,'You're fired.|||That's another silly misconce...,youre fired thats another silly misconception ...


In [13]:
df["type"].value_counts()

type
INFP    1832
INFJ    1470
INTP    1304
INTJ    1091
ENTP     685
ENFP     675
ISTP     337
ISFP     271
ENTJ     231
ISTJ     205
ENFJ     190
ISFJ     166
ESTP      89
ESFP      48
ESFJ      42
ESTJ      39
Name: count, dtype: int64

In [14]:
top_types = [
    "INFP", "INFJ", "INTP", "INTJ",
    "ENTP", "ENFP", "ISTP", "ISFP"
]

In [15]:
df = df[df["type"].isin(top_types)]

In [16]:
df["type"].value_counts()


type
INFP    1832
INFJ    1470
INTP    1304
INTJ    1091
ENTP     685
ENFP     675
ISTP     337
ISFP     271
Name: count, dtype: int64

In [17]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["label"] = le.fit_transform(df["type"])

In [18]:
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(label_mapping)

{'ENFP': np.int64(0), 'ENTP': np.int64(1), 'INFJ': np.int64(2), 'INFP': np.int64(3), 'INTJ': np.int64(4), 'INTP': np.int64(5), 'ISFP': np.int64(6), 'ISTP': np.int64(7)}


In [19]:
df[["type", "label"]].head()

,type,label
0,INFJ,2
1,ENTP,1
2,INTP,5
3,INTJ,4
5,INTJ,4


In [20]:
from sklearn.model_selection import train_test_split

In [21]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["clean_posts"],     # cleaned text
    df["label"],           # encoded labels
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [22]:
print("Train size:", len(train_texts))
print("Validation size:", len(val_texts))

Train size: 6132
Validation size: 1533


In [23]:
import pandas as pd

print(pd.Series(train_labels).value_counts())

label
3    1465
2    1176
5    1043
4     873
1     548
0     540
7     270
6     217
Name: count, dtype: int64


In [24]:
from transformers import DistilBertTokenizer

C:\Users\Dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [25]:
tokenizer = DistilBertTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

In [26]:
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=256
)

In [27]:
val_encodings = tokenizer(
    list(val_texts),
    truncation=True,
    padding=True,
    max_length=256
)

In [28]:
list(train_encodings.keys())

['input_ids', 'attention_mask']

In [29]:
print(train_encodings["input_ids"][0][:20])

[101, 10047, 2019, 4372, 22540, 1998, 1045, 2293, 4372, 2102, 22578, 7916, 7916, 2828, 2070, 2360, 4372, 22540, 2070, 2360]


In [30]:
import torch

In [31]:
class MBTIDataset(torch.utils.data.Dataset):
    
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [32]:
train_dataset = MBTIDataset(train_encodings, train_labels)
val_dataset = MBTIDataset(val_encodings, val_labels)

In [33]:
print(train_dataset[0])

{'input_ids': tensor([  101, 10047,  2019,  4372, 22540,  1998,  1045,  2293,  4372,  2102,
        22578,  7916,  7916,  2828,  2070,  2360,  4372, 22540,  2070,  2360,
         1999, 22540,  2017,  2360,  9686,  2546, 25787,  2140,  1045,  2113,
         2003,  1045, 14396,  2000,  2014,  2926,  1999,  1996,  2927,  2007,
         2014,  7570, 27982,  4606,  2006,  1037,  3167,  3602,  1045,  2113,
         2049,  2066,  2000,  2022,  2058, 21572, 26557,  3064,  8840,  2140,
         2061,  2008, 12991, 13874,  2115,  2063,  2007,  1997,  2607,  1045,
         2812,  1045,  2123,  2102,  7499,  2017,  1045,  2293,  2129,  2017,
         7697,  2619,  2091,  2005, 12991,  3723,  4691,  1998,  2059,  2175,
         2046,  2017,  2219, 23979, 12991, 13874,  1997,  4372, 22540,  2015,
         2074,  8025,  2106,  2017,  2131,  2023, 11654,  2012,  7928,  2500,
         2081,  2008,  2020,  3565,  4895, 10258, 20097,  2075,  1997,  4372,
        22540,  2015,  3426,  2017,  2020,  2428, 

In [34]:
from transformers import DistilBertForSequenceClassification

In [35]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=8
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 757.03it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [36]:
from transformers import TrainingArguments, Trainer

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",     
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [38]:
import transformers
print(transformers.__version__)

5.2.0


In [39]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [40]:
trainer.train()

C:\Users\Dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,1.458197,1.441022
2,1.274355,1.335579


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.52it/s]
C:\Users\Dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.52it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=1534, training_loss=1.4737587795730054, metrics={'train_runtime': 3861.6342, 'train_samples_per_second': 3.176, 'train_steps_per_second': 0.397, 'total_flos': 812377004802048.0, 'train_loss': 1.4737587795730054, 'epoch': 2.0})

In [41]:
trainer.evaluate()

C:\Users\Dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 1.3355792760849,
 'eval_runtime': 103.9686,
 'eval_samples_per_second': 14.745,
 'eval_steps_per_second': 1.847,
 'epoch': 2.0}